# Session 3, Module 09: Logging


This module covers:
- Why logging over print()
- Logging levels and when to use them
- Configuring loggers, handlers, and formatters
- Best practices for data engineering

Data Engineering Context:
Proper logging is essential for monitoring ETL pipelines,
debugging issues, and maintaining audit trails.


In [18]:
import logging
import sys
from datetime import datetime
from pathlib import Path
import tempfile

## Why Logging Over Print?


In [19]:
print("=== Why Logging over print()? ===")

print("""
PRINT LIMITATIONS:
  - No severity levels
  - Hard to turn off/filter
  - No timestamps by default
  - Can't route to files easily
  - Clutters production output

LOGGING BENEFITS:
  - Severity levels (DEBUG, INFO, WARNING, ERROR, CRITICAL)
  - Configurable output (console, file, remote)
  - Timestamps and context automatically
  - Filter by level/logger name
  - Zero changes to code to adjust verbosity
""")

=== Why Logging over print()? ===

PRINT LIMITATIONS:
  - No severity levels
  - Hard to turn off/filter
  - No timestamps by default
  - Can't route to files easily
  - Clutters production output

LOGGING BENEFITS:
  - Severity levels (DEBUG, INFO, WARNING, ERROR, CRITICAL)
  - Configurable output (console, file, remote)
  - Timestamps and context automatically
  - Filter by level/logger name
  - Zero changes to code to adjust verbosity



## Basic Logging


In [3]:
print("\n=== Basic Logging ===")

# Configure basic logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s - %(message)s"
)

# Get a logger
logger = logging.getLogger(__name__)

# Log at different levels
print("\nDemonstrating log levels:")
logger.debug("Debug message - detailed diagnostic info")
logger.info("Info message - confirmation things are working")
logger.warning("Warning message - something unexpected")
logger.error("Error message - something failed")
logger.critical("Critical message - program may crash")

DEBUG - Debug message - detailed diagnostic info
INFO - Info message - confirmation things are working
WARNING - Warning message - something unexpected
ERROR - Error message - something failed
CRITICAL - Critical message - program may crash



=== Basic Logging ===

Demonstrating log levels:


## Logging Levels


In [4]:
print("\n=== Logging Levels ===")

print("""
LEVEL       VALUE   WHEN TO USE
─────────────────────────────────────────────────────────
DEBUG       10      Detailed diagnostic information
                    → Variable values, function entry/exit
                    → Only in development/debugging

INFO        20      Confirmation that things work as expected
                    → Pipeline started, records processed
                    → Normal operations in production

WARNING     30      Something unexpected, but program continues
                    → Deprecated API usage, low disk space
                    → Monitor these in production

ERROR       40      Something failed, but program continues
                    → Failed to process a record, API timeout
                    → Needs attention, may need retry

CRITICAL    50      Program may not be able to continue
                    → Database connection failed, out of memory
                    → Immediate attention required
""")


=== Logging Levels ===

LEVEL       VALUE   WHEN TO USE
─────────────────────────────────────────────────────────
DEBUG       10      Detailed diagnostic information
                    → Variable values, function entry/exit
                    → Only in development/debugging

INFO        20      Confirmation that things work as expected
                    → Pipeline started, records processed
                    → Normal operations in production

WARNING     30      Something unexpected, but program continues
                    → Deprecated API usage, low disk space
                    → Monitor these in production

ERROR       40      Something failed, but program continues
                    → Failed to process a record, API timeout
                    → Needs attention, may need retry

CRITICAL    50      Program may not be able to continue
                    → Database connection failed, out of memory
                    → Immediate attention required



## Configuring Loggers


In [5]:
print("\n=== Configuring Loggers ===")

# Reset logging to demonstrate configuration
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)


def setup_logging(
    level: str = "INFO",
    log_file: str = None,
    format_string: str = None
) -> logging.Logger:
    """
    Set up logging configuration.

    Args:
        level: Logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
        log_file: Optional file path for log output
        format_string: Optional custom format string

    Returns:
        Configured root logger
    """
    # Default format
    if format_string is None:
        format_string = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

    # Create formatter
    formatter = logging.Formatter(format_string, datefmt="%Y-%m-%d %H:%M:%S")

    # Get root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(getattr(logging, level.upper()))

    # Console handler
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    root_logger.addHandler(console_handler)

    # File handler (optional)
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        root_logger.addHandler(file_handler)

    return root_logger


# Set up with custom configuration
temp_log = Path(tempfile.gettempdir()) / "pipeline.log"
setup_logging(level="DEBUG", log_file=str(temp_log))

# Now logging goes to both console and file
logger = logging.getLogger("demo")
logger.info("This goes to console AND file")
logger.debug("So does this debug message")

print(f"\nLog file created at: {temp_log}")


=== Configuring Loggers ===
2026-06-16 17:30:17 - demo - INFO - This goes to console AND file
2026-06-16 17:30:17 - demo - DEBUG - So does this debug message

Log file created at: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/pipeline.log


## Logger Hierarchy


In [6]:
print("\n=== Logger Hierarchy ===")

print("""
Loggers form a hierarchy based on their names:

root
├── etl
│   ├── etl.extract
│   │   ├── etl.extract.api
│   │   └── etl.extract.database
│   ├── etl.transform
│   └── etl.load
└── utils

PROPAGATION:
  - Messages propagate UP the hierarchy
  - etl.extract.api → etl.extract → etl → root
  - Set propagate=False to stop

CONFIGURATION:
  - Configure parent, children inherit settings
  - Override at any level
""")

# Demonstrate hierarchy
parent_logger = logging.getLogger("etl")
child_logger = logging.getLogger("etl.transform")

parent_logger.setLevel(logging.INFO)
# child inherits INFO level

child_logger.info("Message from etl.transform")


=== Logger Hierarchy ===

Loggers form a hierarchy based on their names:

root
├── etl
│   ├── etl.extract
│   │   ├── etl.extract.api
│   │   └── etl.extract.database
│   ├── etl.transform
│   └── etl.load
└── utils

PROPAGATION:
  - Messages propagate UP the hierarchy
  - etl.extract.api → etl.extract → etl → root
  - Set propagate=False to stop

CONFIGURATION:
  - Configure parent, children inherit settings
  - Override at any level

2026-06-16 17:30:46 - etl.transform - INFO - Message from etl.transform


## Format String Placeholders


In [8]:
print("\n=== Format String Placeholders ===")

print("""
COMMON PLACEHOLDERS:
  %(asctime)s      - Timestamp
  %(name)s         - Logger name
  %(levelname)s    - Level (DEBUG, INFO, etc.)
  %(message)s      - The log message
  %(filename)s     - Source file name
  %(lineno)d       - Line number
  %(funcName)s     - Function name
  %(module)s       - Module name
  %(process)d      - Process ID
  %(thread)d       - Thread ID
  %(threadName)s   - Thread name

EXAMPLE FORMATS:

Simple:
  "%(levelname)s - %(message)s"
  → INFO - Pipeline started

Standard:
  "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
  → 2024-01-15 10:30:00 - etl.pipeline - INFO - Pipeline started

Detailed:
  "%(asctime)s [%(levelname)s] %(name)s:%(funcName)s:%(lineno)d - %(message)s"
  → 2024-01-15 10:30:00 [INFO] etl.pipeline:run:42 - Pipeline started

JSON (for log aggregation):
  Use custom formatter or python-json-logger package
""")


=== Format String Placeholders ===

COMMON PLACEHOLDERS:
  %(asctime)s      - Timestamp
  %(name)s         - Logger name
  %(levelname)s    - Level (DEBUG, INFO, etc.)
  %(message)s      - The log message
  %(filename)s     - Source file name
  %(lineno)d       - Line number
  %(funcName)s     - Function name
  %(module)s       - Module name
  %(process)d      - Process ID
  %(thread)d       - Thread ID
  %(threadName)s   - Thread name

EXAMPLE FORMATS:

Simple:
  "%(levelname)s - %(message)s"
  → INFO - Pipeline started

Standard:
  "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
  → 2024-01-15 10:30:00 - etl.pipeline - INFO - Pipeline started

Detailed:
  "%(asctime)s [%(levelname)s] %(name)s:%(funcName)s:%(lineno)d - %(message)s"
  → 2024-01-15 10:30:00 [INFO] etl.pipeline:run:42 - Pipeline started

JSON (for log aggregation):
  Use custom formatter or python-json-logger package



## Handlers — Where Logs Go


In [7]:
print("\n=== Handlers — Where Logs Go ===")

print("""
COMMON HANDLERS:

StreamHandler       - Output to console (stdout/stderr)
FileHandler         - Output to file
RotatingFileHandler - File with size-based rotation
TimedRotatingFileHandler - File with time-based rotation
SocketHandler       - Send to network socket
SMTPHandler         - Send emails for critical errors
HTTPHandler         - Send to HTTP endpoint

EXAMPLE: Rotating file handler
""")

from logging.handlers import RotatingFileHandler, TimedRotatingFileHandler

# Rotating by size
size_rotating_handler = RotatingFileHandler(
    temp_log,
    maxBytes=10*1024*1024,  # 10 MB
    backupCount=5           # Keep 5 backup files
)

# Rotating by time
time_rotating_handler = TimedRotatingFileHandler(
    temp_log,
    when="midnight",        # Rotate at midnight
    interval=1,             # Every 1 day
    backupCount=30          # Keep 30 days
)

print("Handlers configured (not attached to logger in this demo)")


=== Handlers — Where Logs Go ===

COMMON HANDLERS:

StreamHandler       - Output to console (stdout/stderr)
FileHandler         - Output to file
RotatingFileHandler - File with size-based rotation
TimedRotatingFileHandler - File with time-based rotation
SocketHandler       - Send to network socket
SMTPHandler         - Send emails for critical errors
HTTPHandler         - Send to HTTP endpoint

EXAMPLE: Rotating file handler

Handlers configured (not attached to logger in this demo)


## Different Levels For Different Handlers


In [8]:
print("\n=== Different Levels for Different Handlers ===")


def setup_dual_logging():
    """Configure DEBUG to file, INFO to console."""
    logger = logging.getLogger("dual")
    logger.setLevel(logging.DEBUG)  # Capture all levels

    # Console: only INFO and above
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter("%(levelname)s - %(message)s"))

    # File: everything including DEBUG
    file_handler = logging.FileHandler(temp_log)
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(logging.Formatter(
        "%(asctime)s - %(levelname)s - %(message)s"
    ))

    logger.addHandler(console)
    logger.addHandler(file_handler)

    return logger


dual_logger = setup_dual_logging()

print("\nDual logging (DEBUG to file, INFO to console):")
dual_logger.debug("This goes to FILE only")
dual_logger.info("This goes to BOTH")
dual_logger.warning("This also goes to BOTH")


=== Different Levels for Different Handlers ===

Dual logging (DEBUG to file, INFO to console):
2026-06-16 17:33:15 - dual - DEBUG - This goes to FILE only
INFO - This goes to BOTH
2026-06-16 17:33:15 - dual - INFO - This goes to BOTH
WARNING - This also goes to BOTH
2026-06-16 17:33:15 - dual - WARNING - This also goes to BOTH


## Logging In Data Pipelines


In [15]:
print("\n=== Logging in Data Pipelines ===")


class PipelineLogger:
    """
    Structured logging for data pipelines.

    Provides consistent log format with pipeline context.
    """

    def __init__(self, pipeline_name: str):
        self.pipeline_name = pipeline_name
        self.logger = logging.getLogger(f"pipeline.{pipeline_name}")
        self.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    def _format(self, message: str, **context) -> str:
        """Add context to log message."""
        ctx_str = " ".join(f"{k}={v}" for k, v in context.items())
        return f"[{self.run_id}] {message} {ctx_str}".strip()

    def start(self):
        """Log pipeline start."""
        self.logger.info(self._format("Pipeline started"))

    def extract(self, source: str, count: int):
        """Log extraction completion."""
        self.logger.info(self._format(
            "Extraction complete",
            source=source,
            records=count
        ))

    def transform(self, count: int, errors: int = 0):
        """Log transformation completion."""
        self.logger.info(self._format(
            "Transformation complete",
            records=count,
            errors=errors
        ))

    def load(self, destination: str, count: int):
        """Log load completion."""
        self.logger.info(self._format(
            "Load complete",
            destination=destination,
            records=count
        ))

    def error(self, message: str, **context):
        """Log error with context."""
        self.logger.error(self._format(message, **context))

    def finish(self, status: str = "success"):
        """Log pipeline completion."""
        self.logger.info(self._format("Pipeline finished", status=status))


# Usage
print("\nPipeline logging example:")
pl = PipelineLogger("daily_etl")
pl.start()
pl.extract("customers.csv", count=1000)
pl.transform(count=950, errors=50)
pl.load("warehouse.customers", count=950)
pl.finish()


=== Logging in Data Pipelines ===

Pipeline logging example:
2026-06-16 18:00:44 - pipeline.daily_etl - INFO - [20260616_180044] Pipeline started
2026-06-16 18:00:44 - pipeline.daily_etl - INFO - [20260616_180044] Extraction complete source=customers.csv records=1000
2026-06-16 18:00:44 - pipeline.daily_etl - INFO - [20260616_180044] Transformation complete records=950 errors=50
2026-06-16 18:00:44 - pipeline.daily_etl - INFO - [20260616_180044] Load complete destination=warehouse.customers records=950
2026-06-16 18:00:44 - pipeline.daily_etl - INFO - [20260616_180044] Pipeline finished status=success


## Logging Exceptions


In [11]:
print("\n=== Logging Exceptions ===")


def process_data():
    """Demonstrate exception logging."""
    logger = logging.getLogger("exception_demo")

    try:
        # Simulate an error
        result = 1 / 0
    except ZeroDivisionError:
        # Option 1: Log error message only
        logger.error("Division by zero occurred")

        # Option 2: Log with exception info (includes traceback)
        logger.exception("Division by zero with traceback:")
    
print("\nException logging:")
process_data()   


=== Logging Exceptions ===

Exception logging:
2026-06-16 17:36:05 - exception_demo - ERROR - Division by zero occurred
2026-06-16 17:36:05 - exception_demo - ERROR - Division by zero with traceback:
Traceback (most recent call last):
  File "/var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/ipykernel_31501/2602961053.py", line 10, in process_data
    result = 1 / 0
             ~~^~~
ZeroDivisionError: division by zero


Option 3: Log error with exc_info=True (same as exception())
logger.error("Division by zero", exc_info=True)

## Best Practices


In [ ]:
print("\n=== Best Practices ===")

print("""
LOGGING BEST PRACTICES:

1. USE MODULE-LEVEL LOGGERS
   logger = logging.getLogger(__name__)
   # Creates hierarchy matching package structure

2. DON'T LOG SENSITIVE DATA
   # Bad
   logger.info(f"User {user} logged in with password {password}")
   # Good
   logger.info(f"User {user} logged in")

3. USE LAZY FORMATTING
   # Bad (string formatted even if not logged)
   logger.debug(f"Processing {len(large_list)} items")
   # Good (formatted only if DEBUG is enabled)
   logger.debug("Processing %d %b items", len(large_list), )

4. LOG AT APPROPRIATE LEVELS
   DEBUG: Detailed diagnostic
   INFO: Normal operations
   WARNING: Unexpected but handled
   ERROR: Failed but continuing
   CRITICAL: System failure

5. INCLUDE CONTEXT
   logger.info("Record processed", extra={
       "record_id": record["id"],
       "batch_id": batch_id,
   })

6. CONFIGURE AT APPLICATION ENTRY POINT
   # In main.py or __main__.py
   if __name__ == "__main__":
       setup_logging()
       main()

7. USE STRUCTURED LOGGING IN PRODUCTION
      """)


=== Best Practices ===

LOGGING BEST PRACTICES:

1. USE MODULE-LEVEL LOGGERS
   logger = logging.getLogger(__name__)
   # Creates hierarchy matching package structure

2. DON'T LOG SENSITIVE DATA
   # Bad
   logger.info(f"User {user} logged in with password {password}")
   # Good
   logger.info(f"User {user} logged in")

3. USE LAZY FORMATTING
   # Bad (string formatted even if not logged)
   logger.debug(f"Processing {len(large_list)} items")
   # Good (formatted only if DEBUG is enabled)
   logger.debug("Processing %d items", len(large_list))

4. LOG AT APPROPRIATE LEVELS
   DEBUG: Detailed diagnostic
   INFO: Normal operations
   ERROR: Failed but continuing
   CRITICAL: System failure

5. INCLUDE CONTEXT
   logger.info("Record processed", extra={
       "record_id": record["id"],
       "batch_id": batch_id,
   })

6. CONFIGURE AT APPLICATION ENTRY POINT
   # In main.py or __main__.py
   if __name__ == "__main__":
       setup_logging()
       main()

7. USE STRUCTURED LOGGING IN 

Consider python-json-logger for log aggregation
Easy to parse by ELK, Splunk, CloudWatch

## Summary


In [16]:
print("\n=== Summary ===")
print("""
Logging Key Points:

BASICS:
  import logging
  logger = logging.getLogger(__name__)
  logger.info("Message")

LEVELS (low to high):
  DEBUG → INFO → WARNING → ERROR → CRITICAL

CONFIGURATION:
  logging.basicConfig(level=logging.INFO, format="...")

COMPONENTS:
  - Logger: What you call to log
  - Handler: Where logs go (console, file, network)
  - Formatter: How logs look
  - Filter: Which logs to process

BEST PRACTICES:
  - Use module-level loggers
  - Don't log sensitive data
  - Use lazy formatting (%, not f-strings)
  - Log at appropriate levels
  - Include context with messages
  - Configure at entry point
""")


=== Summary ===

Logging Key Points:

BASICS:
  import logging
  logger = logging.getLogger(__name__)
  logger.info("Message")

LEVELS (low to high):
  DEBUG → INFO → WARNING → ERROR → CRITICAL

CONFIGURATION:
  logging.basicConfig(level=logging.INFO, format="...")

COMPONENTS:
  - Logger: What you call to log
  - Handler: Where logs go (console, file, network)
  - Formatter: How logs look
  - Filter: Which logs to process

BEST PRACTICES:
  - Use module-level loggers
  - Don't log sensitive data
  - Use lazy formatting (%, not f-strings)
  - Log at appropriate levels
  - Include context with messages
  - Configure at entry point

